In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import re

# Configurar opciones de Selenium
options = Options()
#options.add_argument("--headless")  # Activa si quieres modo sin ventana
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

def extraer_info_producto_movistar(product, fecha_consulta):
    """
    Extrae información de cada producto de Movistar
    """
    try:
        # Encontrar el contenedor padre que tiene todos los datos del producto
        contenedor_producto = product
        parent = product.find_parent()
        
        # Buscar hasta 5 niveles hacia arriba para encontrar el contenedor completo
        for _ in range(5):
            if parent:
                precio_test = parent.find("p", class_="grid__terminal-prepago-precio")
                if precio_test:
                    contenedor_producto = parent
                    break
                parent = parent.find_parent()
        
        # Nombre del producto
        nombre = ""
        nombre_tag = product.find("h2", class_="grid__title")
        if nombre_tag:
            nombre = nombre_tag.get_text(strip=True)
        
        if not nombre:
            onclick_text = product.get("onclick", "")
            match = re.search(r"taggearClick\('([^']*)'", onclick_text)
            if match:
                nombre = match.group(1)
        
        # URL del producto
        url_producto = ""
        if product.get("href"):
            href = product["href"]
            if href.startswith("http"):
                url_producto = href
            elif href.startswith("/"):
                url_producto = "https://tienda.movistar.com.mx" + href
        
        # SKU
        sku = ""
        onclick_text = product.get("onclick", "")
        if onclick_text:
            match = re.search(r"taggearClick\([^,]*,[^,]*,[^,]*,[^,]*,[^,]*,[^,]*,[^,]*,[^,]*,'([^']*)'", onclick_text)
            if match:
                sku = match.group(1)
        
        if not sku:
            sku = product.get("data-dl-product-view-detail", "")
        
        # Marca
        marca = ""
        if nombre:
            nombre_lower = nombre.lower()
            marcas_conocidas = [
                'iphone', 'apple', 'samsung', 'galaxy', 'xiaomi', 'redmi', 'huawei', 
                'oneplus', 'motorola', 'moto', 'nokia', 'oppo', 'vivo', 'realme', 
                'honor', 'tcl', 'lg', 'sony', 'google', 'pixel', 'nothing', 'zte',
                'infinix', 'tecno', 'poco'
            ]
            
            for marca_buscar in marcas_conocidas:
                if marca_buscar in nombre_lower:
                    if marca_buscar in ['iphone', 'apple']:
                        marca = "APPLE"
                    elif marca_buscar in ['samsung', 'galaxy']:
                        marca = "SAMSUNG"
                    elif marca_buscar in ['xiaomi', 'redmi', 'poco']:
                        marca = "XIAOMI"
                    elif marca_buscar in ['motorola', 'moto']:
                        marca = "MOTOROLA"
                    elif marca_buscar in ['google', 'pixel']:
                        marca = "GOOGLE"
                    else:
                        marca = marca_buscar.upper()
                    break
        
        # Color
        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            colores = [
                'negro', 'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
                'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple',
                'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver',
                'violeta', 'violet', 'naranja', 'orange', 'celeste', 'turquesa',
                'midnight', 'starlight', 'sierra', 'alpine', 'phantom', 'titanio', 
                'cyan', 'café', 'plata', 'lila', 'beige', 'coral', 'lavanda',
                'crema', 'bronce', 'arena', 'menta'
            ]
            
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break
        
        # Capacidad
        capacidad = ""
        if nombre:
            capacidad_match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', nombre, re.IGNORECASE)
            if capacidad_match:
                numero = capacidad_match.group(1)
                unidad = capacidad_match.group(2).upper()
                capacidad = f"{numero} {unidad}"
        
        # Función para limpiar precios
        def limpiar_precio_texto(precio_texto):
            if precio_texto:
                precio_limpio = re.sub(r'[^\d,.]', '', precio_texto.replace('$', '').replace('MXN', ''))
                return precio_limpio.strip()
            return ""
        
        # Precios - buscar en el contenedor padre
        precio_original = ""
        precio_descuento = ""
        
        precio_tachado_tag = contenedor_producto.find("p", class_="grid__terminal-prepago-precio-tachado")
        if precio_tachado_tag:
            precio_text = precio_tachado_tag.get_text(strip=True)
            precio_original = limpiar_precio_texto(precio_text)
        
        precio_actual_tag = contenedor_producto.find("p", class_="grid__terminal-prepago-precio")
        if precio_actual_tag:
            precio_text = precio_actual_tag.get_text(strip=True)
            precio_descuento = limpiar_precio_texto(precio_text)
        
        # Lógica de precios
        if not precio_original and precio_descuento:
            precio_original = precio_descuento
            precio_descuento = "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        elif not precio_original and not precio_descuento:
            precio_original = "No inventory"
            precio_descuento = "No discount"
        
        # Porcentaje de descuento
        porcentaje_str = "No discount"
        porcentaje_tag = contenedor_producto.find("p", class_="grid__terminal-prepago-porcentaje-descuento")
        if porcentaje_tag:
            porcentaje_text = porcentaje_tag.get_text(strip=True)
            porcentaje_match = re.search(r'(\d+)%', porcentaje_text)
            if porcentaje_match:
                porcentaje_str = porcentaje_match.group(0)
        
        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku,
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }
        
    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def obtener_total_paginas(soup):
    """
    Detecta el número total de páginas desde el HTML
    """
    # Método 1: Buscar "Elementos X-Y de Z"
    toolbar = soup.find("p", class_="toolbar-amount")
    if toolbar:
        texto = toolbar.get_text(strip=True)
        match = re.search(r'de\s+(\d+)', texto)
        if match:
            total_productos = int(match.group(1))
            productos_por_pagina = 12
            total_paginas = (total_productos + productos_por_pagina - 1) // productos_por_pagina
            print(f"Total de productos detectados: {total_productos}")
            print(f"Total de páginas calculadas: {total_paginas}")
            return total_paginas
    
    # Método 2: Buscar el número de página más alto en la paginación
    page_links = soup.select("ul.items.pages-items li.item a")
    if page_links:
        max_page = 1
        for link in page_links:
            page_text = link.get_text(strip=True)
            if page_text.isdigit():
                max_page = max(max_page, int(page_text))
        if max_page > 1:
            print(f"Total de páginas detectadas por paginación: {max_page}")
            return max_page
    
    # Método 3: Buscar cualquier texto que contenga el total
    all_text = soup.get_text()
    match = re.search(r'(\d+)-(\d+)\s+de\s+(\d+)', all_text)
    if match:
        total_productos = int(match.group(3))
        productos_por_pagina = 12
        total_paginas = (total_productos + productos_por_pagina - 1) // productos_por_pagina
        print(f"Total productos (método 3): {total_productos}, páginas: {total_paginas}")
        return total_paginas
    
    return None

def scrape_movistar_smartphones():
    """
    Función principal para hacer scraping de smartphones en Movistar.
    Navega directamente por URL: telefonos.html?p=1, ?p=2, ?p=3, etc.
    """
    base_url = "https://tienda.movistar.com.mx/telefonos.html"
    
    print("=" * 60)
    print("INICIANDO SCRAPING DE MOVISTAR")
    print("=" * 60)
    
    driver = webdriver.Chrome(options=options)
    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    productos_procesados = set()
    max_paginas = 30  # Límite de seguridad
    
    try:
        # 1) Cargar primera página para detectar total de páginas
        print(f"Cargando página inicial: {base_url}")
        driver.get(base_url)
        time.sleep(5)
        
        # Scroll inicial para cargar todo
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)
        
        soup_inicial = BeautifulSoup(driver.page_source, "html.parser")
        total_paginas = obtener_total_paginas(soup_inicial)
        
        if total_paginas:
            max_paginas = min(total_paginas, 30)
            print(f"Se procesarán {max_paginas} páginas")
        else:
            # Si no detecta, intentar con un número alto y parar cuando no haya productos
            max_paginas = 20
            print(f"No se detectó total de páginas, se intentarán hasta {max_paginas}")
        
        # 2) Recorrer TODAS las páginas por URL directa
        paginas_vacias_consecutivas = 0
        
        for page_number in range(1, max_paginas + 1):
            print(f"\n{'='*50}")
            print(f"PÁGINA {page_number} de {max_paginas}")
            print(f"{'='*50}")
            
            # Construir URL de la página
            if page_number == 1:
                page_url = base_url
            else:
                page_url = f"{base_url}?p={page_number}"
            
            # Solo navegar si no es la primera página (ya la cargamos)
            if page_number > 1:
                print(f"Navegando a: {page_url}")
                driver.get(page_url)
                
                # Esperar carga completa
                WebDriverWait(driver, 15).until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                )
                time.sleep(3)
                
                # Scroll para cargar contenido dinámico
                for _ in range(3):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)
            
            # Obtener HTML de la página actual
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Buscar productos
            products = soup.find_all("a", {"data-dl-product-view-detail": True})
            print(f"Productos encontrados en página {page_number}: {len(products)}")
            
            # Si no hay productos, contar páginas vacías
            if len(products) == 0:
                paginas_vacias_consecutivas += 1
                print(f"Página sin productos (consecutivas: {paginas_vacias_consecutivas})")
                if paginas_vacias_consecutivas >= 2:
                    print("2 páginas vacías consecutivas. Finalizando...")
                    break
                continue
            else:
                paginas_vacias_consecutivas = 0
            
            # Debug en primera página
            if page_number == 1:
                for i, product in enumerate(products[:2]):
                    print(f"\n--- Debug Producto {i+1} ---")
                    nombre_tag = product.find("h2", class_="grid__title")
                    if nombre_tag:
                        print(f"Nombre: {nombre_tag.get_text(strip=True)}")
                    sku = product.get("data-dl-product-view-detail", "")
                    print(f"Data SKU: {sku}")
                    
                    parent = product.find_parent()
                    for _ in range(5):
                        if parent:
                            precio_actual = parent.find("p", class_="grid__terminal-prepago-precio")
                            if precio_actual:
                                print(f"Precio: {precio_actual.get_text(strip=True)}")
                                break
                            parent = parent.find_parent()
            
            # Extraer información de productos
            productos_pagina_actual = 0
            
            for product in products:
                sku_temp = product.get("data-dl-product-view-detail", "")
                
                # Evitar duplicados
                if sku_temp and sku_temp in productos_procesados:
                    continue
                    
                info = extraer_info_producto_movistar(product, fecha_consulta)
                if info and info["Nombre"] and len(info["Nombre"]) > 5 and info["Marca"]:
                    data.append(info)
                    if sku_temp:
                        productos_procesados.add(sku_temp)
                    productos_pagina_actual += 1
                    print(f"  ✅ {info['Nombre'][:55]} | SKU: {info['SKU']} | ${info['Precio_original']}")
            
            print(f"\nNuevos en página {page_number}: {productos_pagina_actual}")
            print(f"Total acumulado: {len(data)}")
        
        print(f"\n{'='*60}")
        print(f"SCRAPING COMPLETADO")
        print(f"Páginas procesadas: {page_number}")
        print(f"Productos únicos extraídos: {len(data)}")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()
    
    return data

# Ejecutar el scraping
resultados_movistar = scrape_movistar_smartphones()

# Exportar a CSV con limpieza de datos
if resultados_movistar:
    df_movistar = pd.DataFrame(resultados_movistar)
    
    print(f"\nProductos antes de limpiar: {len(df_movistar)}")
    
    # 1. Eliminar filas donde el nombre esté vacío o sea muy corto
    df_movistar = df_movistar[df_movistar['Nombre'].str.len() > 5]
    
    # 2. Eliminar filas donde la marca esté vacía
    df_movistar = df_movistar[df_movistar['Marca'].str.len() > 0]
    
    # 3. Eliminar duplicados basados en SKU
    df_movistar = df_movistar.drop_duplicates(subset=['SKU'], keep='first')
    
    # 4. Reordenar por marca y nombre
    df_movistar = df_movistar.sort_values(['Marca', 'Nombre'])
    
    # 5. Resetear índice
    df_movistar = df_movistar.reset_index(drop=True)
    
    print(f"Productos después de limpiar: {len(df_movistar)}")
    
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"movistar_smartphones_completo_{fecha_hoy}.csv"
    df_movistar.to_csv(nombre_archivo, index=False, encoding="utf-8")
    print(f"\nDatos exportados a {nombre_archivo}")
    print(f"Total de productos únicos: {len(df_movistar)}")
    
    # Mostrar estadísticas
    print("\n=== ESTADÍSTICAS FINALES ===")
    print(f"Total de productos: {len(df_movistar)}")
    print(f"\nProductos por marca:")
    print(df_movistar['Marca'].value_counts())
    print(f"\nProductos con descuento: {len(df_movistar[df_movistar['Porcentaje_descuento'] != 'No discount'])}")
    
    print("\n=== PRIMEROS 10 PRODUCTOS ===")
    print(df_movistar.head(10))
    
else:
    print("No se encontraron productos válidos.")

INICIANDO SCRAPING DE MOVISTAR
Cargando página inicial: https://tienda.movistar.com.mx/telefonos.html
Total productos (método 3): 121, páginas: 11
Se procesarán 11 páginas

PÁGINA 1 de 11
Productos encontrados en página 1: 24

--- Debug Producto 1 ---
Data SKU: 3967
Precio: $35,999

--- Debug Producto 2 ---
Nombre: Xiaomi 17 Ultra 512 GB Negro con Photography Kit Pro + Xiaomi Sound Party Negro
Data SKU: 3967
Precio: $35,999
  ✅ Xiaomi 17 Ultra 512 GB Negro con Photography Kit Pro +  | SKU: TMLMXXIX17UBNE12 | $35,999
  ✅ Samsung Galaxy S26 Ultra 512 GB 5G Negro | SKU: TMLMXSAS26ULNE12 | $37,999
  ✅ Samsung Galaxy S26 Plus 512 GB 5G Violeta | SKU: TMLMXSAS26PLVI12 | $33,499
  ✅ Samsung Galaxy S26 Plus 512 GB 5G Negro | SKU: TMLMXSAS26PLNE12 | $33,499
  ✅ Apple iPhone 17 Pro Max 256 GB Plata | SKU: TMLMXAPI17PMPL56 | $30,999
  ✅ Apple iPhone 17e 256 GB Negro | SKU: TMLMXAPIP17ENE56 | $14,999
  ✅ Xiaomi Redmi Note 15 256 GB Negro con Licuadora Portáti | SKU: TMLMXXIN15BMNE56 | $5,999.00
  